# Day 1:
**Imbalanced Data:** SMOTE, undersampling, oversampling (avoiding data leakage), class weights.

In [3]:
# Imports:
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier

### Question 1:
- Scenario: You are building a fraud detection model using the data above. You train a standard RandomForestClassifier. Your model predicts that every single transaction is legitimate (Class 0). Because only 1% of the data is actual fraud, your model achieves 99% accuracy.

1. Part 1 (Theory): Why is "accuracy" a terrible metric for this scenario, and what are two better metrics you would look at to evaluate fraud detection? Conceptually, how does applying "Class Weights" to an algorithm help solve this problem without actually changing the underlying dataset?

2. Part 2 (Implementation): Write the Scikit-Learn code to instantiate a RandomForestClassifier that automatically balances the class weights. (You don't need to fit or predict, just show me the model instantiation).

In [2]:
# Create a heavily imbalanced dataset (99% class 0, 1% class 1)
X, y = make_classification(n_samples=10000, n_features=5, weights=[0.99], random_state=42)
X_train = pd.DataFrame(X, columns=[f'feature_{i}' for i in range(5)])
y_train = pd.Series(y, name='target')

**Why is "accuracy" a terrible metric for this scenario, and what are two better metrics you would look at to evaluate fraud detection? Conceptually, how does applying "Class Weights" to an algorithm help solve this problem without actually changing the underlying dataset?**

- Accuracy takes into account only the correct predictions made vs false predictions made. For an imabalnced dataset, if one class constitutes 99% of data, simply predicting that class will get the model accuracy of 99%, without ever predicting the second class for any data point. We can replace that model with a simple line of code assigning same class for every data point without any calculations and modelling.

- Better Matrix to consider here would be Recall (Out of all positive cases, how many did we catch), and f1-score which is a Harmonic Mean of Precision and Recall, considering both the classes.

- Class Weights work as kind of a penulty for model while predicting different classes. Usually, all classes share the same weight. But using class weight parameter, we can put more weight on less frequent class to penalize model more for not predicting that class correctly, making model consider the less frequent class more seriously.

**Write the Scikit-Learn code to instantiate a RandomForestClassifier that automatically balances the class weights. (You don't need to fit or predict, just show me the model instantiation).**

In [4]:
rf_model= RandomForestClassifier(class_weight= 'balanced')

### Gold Standard: Class Weights & Imbalanced Metrics

*   **Explanation:** Class weighting modifies the algorithm's cost function so that misclassifying the minority class incurs a disproportionately higher penalty. 
*   **Core Intuition:** Instead of generating synthetic data or deleting real data to balance the classes, you are mathematically forcing the model to pay more attention to the rare events. To evaluate if it worked, you must drop Accuracy and use Precision, Recall, or the F1-Score (which balances Precision and Recall).
*   **Pros and Cons:**
    *   *Pros:* Extremely fast to implement (just one parameter change). Does not alter the original training data or increase the dataset size, keeping memory usage low.
    *   *Cons:* May not perform as well as synthetic data generation techniques (like SMOTE) when the minority class is extremely small or lacks distinct decision boundaries.
*   **Edge Cases:** The `'balanced'` heuristic automatically calculates weights inversely proportional to class frequencies: `n_samples / (n_classes * np.bincount(y))`. If you need custom penalties (e.g., in medical diagnostics where a false negative is catastrophic), you can pass a custom dictionary instead: `class_weight={0: 1, 1: 50}`.
*   **Standard Code Syntax:**
```python
from sklearn.ensemble import RandomForestClassifier

# Automatically balance based on class frequencies
rf_model = RandomForestClassifier(class_weight='balanced', random_state=42)